# Digital Cash Management Forecast: inference load test (sandbox only)

Checks that the forecast scales to the bank's number of clients, and measures it: the benchmark of a Dataiku
server. **Never in the production project**: it replaces `clients` and `daily_flows` with fake data.

Run it in a sandbox project where the forecast is set up (`build_forecast_project.ipynb` done, so there is a
production model). It:

1. writes fake `clients` and `daily_flows` for `N_CLIENTS` clients (random amounts, the real columns);
2. runs the recipe `final_forecast` alone and shows its row of `inference_log`: rows, chunks of clients, seconds
   reading and forecasting, and the most memory it used (`peak_memory_mb`);
3. puts the real data back (scenario **Full rebuild of the daily flows**, then **Nightly batch**).

Inference forecasts `CLIENTS_PER_CHUNK` clients at a time (project variable `inference_clients_per_chunk`): a
bigger chunk is faster but needs more memory. Try a few values on the bank's server and keep the best one.

In [ ]:
N_CLIENTS = 420_000          # fake business clients
MIN_MONTHS, MAX_MONTHS = 6, 36   # each client's history: a random number of months up to the cut-off month (some
                                 # under 12, so a few get NOT_ENOUGH_HISTORY)
ROWS_PER_MONTH = 16          # daily_flows rows per client and month (days with postings x categories)
CLIENTS_PER_CHUNK = 5000     # sets the project variable inference_clients_per_chunk for the test

In [ ]:
import time

import dataiku
import numpy as np
import pandas as pd

from cashflow import cutoff_month, read_dataset
from forecast import active_model_id
from warehouse import CATEGORY_OF_TYPE, OTHER

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())
cutoff = cutoff_month()
print(f"Production model {active_model_id()}, cut-off month {cutoff:%Y-%m}")   # stops here if there is none yet

## 1. Fake data

In [ ]:
# Fake clients (copies of the real ones' rows with new client numbers, so the columns are the same) and fake daily
# flows (random amounts), written straight to clients and daily_flows, in batches of clients. staging_log is emptied:
# the next staging run then stops and asks for the scenario Full rebuild, so fake and real data never mix.
started = time.time()
rng = np.random.default_rng(7)
categories = np.array(sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER}))
base = read_dataset("clients")
fake = base.sample(N_CLIENTS, replace=True, random_state=7).reset_index(drop=True)
fake["client_no"] = [f"LT{i:07d}" for i in range(N_CLIENTS)]
dataiku.Dataset("clients").write_with_schema(fake)
for name in ("daily_flows", "staging_log"):
    project.get_dataset(name).clear()

daily_flows = dataiku.Dataset("daily_flows")
last = np.datetime64(f"{cutoff:%Y-%m}", "M")
rows, writer = 0, None
for start in range(0, N_CLIENTS, 10_000):
    numbers = fake["client_no"].to_numpy()[start:start + 10_000]
    n = len(numbers)
    per_client = rng.integers(MIN_MONTHS, MAX_MONTHS + 1, n) * ROWS_PER_MONTH
    who = np.repeat(np.arange(n), per_client)
    position = np.arange(len(who)) - np.repeat(np.cumsum(per_client) - per_client, per_client)
    month = (last - position // ROWS_PER_MONTH).astype("datetime64[D]")
    size = rng.lognormal(9, 1.2, n)[who]                       # each client's typical amount (SAR)
    batch = pd.DataFrame({
        "client_no": numbers[who],
        "day": (month + rng.integers(0, 28, len(who))).astype("datetime64[ns]"),
        "category": rng.choice(categories, len(who)),
        "money_in": np.where(rng.random(len(who)) < 0.6, size * rng.lognormal(0, 0.6, len(who)), 0.0),
        "money_out": np.where(rng.random(len(who)) < 0.6, size * rng.lognormal(0, 0.6, len(who)), 0.0),
        "postings": rng.integers(1, 6, len(who))})
    batch = (batch.groupby(["client_no", "day", "category"], as_index=False).sum()
             .round({"money_in": 2, "money_out": 2}))
    if writer is None:
        daily_flows.write_schema_from_dataframe(batch)
        writer = daily_flows.get_writer()
    writer.write_dataframe(batch)
    rows += len(batch)
writer.close()
print(f"{N_CLIENTS:,} fake clients, {rows:,} daily_flows rows written in {(time.time() - started) / 60:.1f} minutes")

## 2. Forecast and measure

In [ ]:
# Runs the recipe final_forecast alone (the forecast, its archive and the inference log), as the Nightly batch does
# after staging, and shows its row of inference_log: rows, chunks, seconds per step and the most memory it used.
variables = project.get_variables()
variables["standard"]["inference_clients_per_chunk"] = CLIENTS_PER_CHUNK
project.set_variables(variables)
started = time.time()
project.new_job("NON_RECURSIVE_FORCED_BUILD").with_output("inference_log").start_and_wait()
print(f"final_forecast took {(time.time() - started) / 60:.1f} minutes (with Dataiku's job start)")
log = dataiku.Dataset("inference_log").get_dataframe()
display(log.sort_values("run_at").tail(1).T)

## 3. Put the real data back

In [ ]:
# The real data back, as the scenario Full rebuild of the daily flows does: empty daily_flows and staging_log, read
# the whole history from the warehouse again (clients and daily_flows), then forecast the real clients.
from warehouse import clear_daily_flows

clear_daily_flows()
project.new_job("RECURSIVE_FORCED_BUILD").with_output("daily_flows").start_and_wait()
project.new_job("NON_RECURSIVE_FORCED_BUILD").with_output("inference_log").start_and_wait()
print("Real data and forecast back:", len(dataiku.Dataset("clients").get_dataframe()), "clients")